# Task #5: answer metrics on the dev split (GPU)

Runs `compare_configurations.py` with the Qwen generator for all four setups on the 118 dev questions: accuracy, citation metrics, unsupported rate, and median/p95 latency.

**Before you start:** Runtime → Change runtime type → **T4 GPU**.

Results go to your Google Drive, so if Colab disconnects you lose nothing: run every cell again from the top and it continues where it stopped (answers already saved are not regenerated). Expect a few hours for the full run.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/btt_task5_results'   # everything is saved here
BRANCH = 'main'                                     # use 'task-5-answer-metrics' until that PR is merged
import os; os.makedirs(OUT, exist_ok=True)

In [ ]:
!rm -rf /content/repo && git clone -q --depth 1 -b {BRANCH} https://github.com/Break-Through-Tech/Retail-1B-grounded-financial-document-qa-with-rag.git /content/repo
%cd /content/repo
!pip install -q faiss-cpu
!python dense_index.py build

Check the GPU and record it next to the results, because latency numbers only make sense with the hardware they were measured on.

In [ ]:
import torch, subprocess
assert torch.cuda.is_available(), 'No GPU: Runtime -> Change runtime type -> T4 GPU'
gpu = torch.cuda.get_device_name(0); print(gpu)
open(f'{OUT}/hardware.txt', 'w').write(gpu + '\n')

**1. Quick trial (3 questions, separate folder).** If this fails, stop and fix it before the long run.

In [ ]:
!python compare_configurations.py --limit 3 --output-dir /content/trial

**2. Full run.** Re-run this cell after any disconnect; it resumes.

In [ ]:
!python compare_configurations.py --output-dir {OUT}

**3. Results.**

In [ ]:
import pandas as pd
from IPython.display import Image, display
pd.set_option('display.width', 250)
display(pd.read_csv(f'{OUT}/retrieval_metrics.csv').round(3))
display(pd.read_csv(f'{OUT}/answer_metrics.csv').round(3))
by_group = pd.read_csv(f'{OUT}/answer_metrics_by_group.csv').round(3)
display(by_group[by_group.group_by == 'question_type'])
display(Image(f'{OUT}/answer_metrics.png'))

**4. Pick the setups to grade by hand.** Look at `answer_accuracy_strict` and the retrieval table, choose the best two, and put them below. This re-run is fast: nothing is regenerated, it only rewrites `answers_to_review.csv` with those setups' free-text answers.

Then open `answers_to_review.csv` (in your Drive folder) in Google Sheets, write `1` (correct) or `0` (wrong) in the `correct` column, and download it back to the same Drive folder as **CSV** with the same name.

In [ ]:
REVIEW = 'dense hybrid'   # change to your two best setups
!python compare_configurations.py --output-dir {OUT} --review-configs {REVIEW}

**5. After grading:** re-run with your labels. Graded accuracy and strict accuracy should now match for the setups you graded.

In [ ]:
!python compare_configurations.py --output-dir {OUT} --review-configs {REVIEW} --manual-labels {OUT}/answers_to_review.csv
display(pd.read_csv(f'{OUT}/answer_metrics.csv').round(3))